# Daily News Summarizer

This is a simple daily news summarizer of the Indian newpaper The Hindu. It uses a simple web scrapper and summarizes it using the llama3.2 model done as a part of day 2 homework.

In [1]:
import requests
from openai import OpenAI
from IPython.display import HTML

### Scrapper code

In [9]:
from bs4 import BeautifulSoup

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"}

def fetch_website_info(url):
    #"truncate to 2000 characters"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content,"html.parser")
    title = soup.title.string if soup.title else "No title found"

    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [10]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [11]:
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [12]:
system_prompt = """You are an assistant journalist. You are tasked with summarizing any important news headline in the current day's newspaper. Respond in html.  Summarize the info in separate headings and paragraphs. Heading would be the headline and a small description of it as the paragraph. Skip any advertisements or description about the newpaper or anything that might not be news"""

user_prompt = """Here is the website information for the newspaper, the hindu. Include atleast 8 headlines"""


In [13]:
def headline_summarizer(url):
    summary = fetch_website_info(url)
    messages = [{"role":"system", "content":system_prompt},
            {"role":"user", "content":user_prompt + summary}]
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    return HTML(response.choices[0].message.content)

In [ ]:
url = "https://www.thehindu.com/"
headline_summarizer(url)